# PolyGuard Finance: Original and Rewritten Training Text

## 1. Goal and setup

This pilot compares 0%, 50%, and 100% replacement of unsafe training examples
with their stored rewritten versions. Base cases, class balance, model settings,
and the decision threshold remain fixed. Safe examples are unchanged.

The main outcomes are unsafe recall (higher is better) and safe false-positive
rate (lower is better). AUC is a secondary diagnostic.

This is an exploratory, single-seed pilot. Both original and rewritten examples
are synthetic. Existing rewrites are not independently verified as perfectly
meaning-preserving.

Use a CPU runtime and run cells in order. This notebook never uploads to or
modifies Hugging Face or GitHub.

In [20]:
%pip -q install numpy==2.2.6 pandas==2.2.3 scikit-learn==1.6.1 matplotlib==3.10.3 requests==2.32.4

## 2. Configuration

Read the pinned original dataset. All generated files remain under assignment/4.

The threshold is fixed at 0.5 before evaluation. It is not selected using the
test results.


In [21]:
import hashlib, json, random, shutil, sys, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests, sklearn

from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score

REPO = "Virtue-AI-HUB/PolyGuard"
REVISION = "fa5b3948e6e23f0611e2382561c1434971550b65"

SEED, THRESHOLD = 42, 0.5
RATIOS = [0.0, 0.5, 1.0]

ROOT = Path("assignment/4/polyguard_colab").resolve()
DATA, RESULTS, RAW = ROOT / "data", ROOT / "results", ROOT / "raw"

for folder in [DATA, RESULTS, RAW]:
    folder.mkdir(parents=True, exist_ok=True)

def normalize(text):
    return " ".join(
        unicodedata.normalize("NFKC", text).casefold().split()
    )

def digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

print("Output folder:", ROOT)
print("scikit-learn:", sklearn.__version__)

Output folder: /content/assignment/4/polyguard_colab
scikit-learn: 1.6.1


## 3. Load financial input cases

Only finance/Input is loaded. Labels come from safe/unsafe filenames.

Each row keeps the original and rewritten versions together. Safe examples
have no stored rewrite.

The raw release contains literal line breaks inside some JSON strings.
An object decoder is used instead of silently dropping malformed JSONL lines.

In [22]:
def decode_objects(text):
    decoder = json.JSONDecoder(strict=False)
    position = 0

    while position < len(text):
        while position < len(text) and text[position].isspace():
            position += 1

        if position == len(text):
            break

        record, position = decoder.raw_decode(text, position)
        yield record

records, raw_manifest = [], []

for source in ["ALT", "BIS", "FINRA", "OECD", "USDT"]:
    for label, name in [
        (0, "data_safe.jsonl"),
        (1, "data_unsafe.jsonl")
    ]:
        relative = f"finance/Input/{source}/{name}"
        url = (
            f"https://huggingface.co/datasets/{REPO}"
            f"/resolve/{REVISION}/{relative}"
        )

        response = requests.get(url, timeout=90)
        response.raise_for_status()
        content = response.content

        (RAW / f"{source}_{name}").write_bytes(content)
        raw_manifest.append({
            "file": relative,
            "sha256": hashlib.sha256(content).hexdigest()
        })

        for item in decode_objects(content.decode("utf-8")):
            original = item["original instance"]
            rewrite = item["rephrased instance"]

            if label == 0:
                assert normalize(rewrite) in [
                    "", "not applicable", "n/a", "none"
                ]
                rewrite = None
            else:
                assert isinstance(rewrite, str)
                assert normalize(rewrite) not in [
                    "", "not applicable", "n/a", "none"
                ]

            records.append({
                "case_id": digest(f"{source}\n{label}\n{original}"),
                "rule_id": digest(normalize(item["rule"])),
                "source": source,
                "label": label,
                "original_text": original,
                "rewritten_text": rewrite
            })

df = pd.DataFrame(records)

print("Loaded cases:", len(df))
display(
    pd.crosstab(df["source"], df["label"])
    .rename(columns={0: "safe", 1: "unsafe"})
)

Loaded cases: 7060


label,safe,unsafe
source,,
ALT,370,370
BIS,455,455
FINRA,1500,1500
OECD,775,775
USDT,430,430


## 4. Validate and split by rule

Split rules approximately 70%/15%/15%, then assign all cases and versions of a
rule to the same split.

Expected sizes:
- Train: 4,940 cases
- Validation: 1,050 cases
- Test: 1,070 cases

These are exploratory splits: this dataset was inspected before this notebook.
No expert label-accuracy audit is claimed.

In [23]:
assert len(df) == 7060
assert df["case_id"].is_unique
assert df["label"].value_counts().to_dict() == {0: 3530, 1: 3530}

assert df["original_text"].map(
    lambda x: isinstance(x, str) and bool(x.strip())
).all()

assert df.loc[df.label == 0, "rewritten_text"].isna().all()
assert df.loc[df.label == 1, "rewritten_text"].notna().all()
assert df.groupby(["rule_id", "label"]).size().eq(5).all()

rules = sorted(df["rule_id"].unique())
assert len(rules) == 706

random.Random(SEED).shuffle(rules)

n_train = int(0.70 * len(rules))
n_val = int(0.15 * len(rules))

rule_sets = {
    "train": set(rules[:n_train]),
    "validation": set(rules[n_train:n_train + n_val]),
    "test": set(rules[n_train + n_val:])
}

assert all(
    not rule_sets[a] & rule_sets[b]
    for a, b in [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test")
    ]
)

membership = {
    rule: split
    for split, group in rule_sets.items()
    for rule in group
}

df["split"] = df["rule_id"].map(membership)

splits = {
    name: df[df.split == name].copy()
    for name in rule_sets
}

for name, frame in splits.items():
    path = DATA / f"{name}.jsonl"
    frame.to_json(
        path, orient="records", lines=True, force_ascii=True
    )
    assert (
        pd.read_json(path, lines=True).case_id.tolist()
        == frame.case_id.tolist()
    )

display(
    pd.crosstab(df["split"], df["label"])
    .rename(columns={0: "safe", 1: "unsafe"})
)

print(
    "Checks passed: unique cases, complete text, "
    "balanced labels, and zero rule overlap."
)

label,safe,unsafe
split,,
test,535,535
train,2470,2470
validation,525,525


Checks passed: unique cases, complete text, balanced labels, and zero rule overlap.


## 5. Build training conditions and fit models

Compare 0%, 50%, and 100% replacement among unsafe training cases only.

Every condition uses the same base cases and safe texts. Replacement subsets
are nested using a shared random ordering.

Only selected text is used as a feature. IDs, rules, sources, labels, and rewrite
availability are not input features. TF-IDF is fitted on training text only.

In [ ]:
train = splits["train"]

unsafe_ids = sorted(
    train.loc[train.label == 1, "case_id"]
)
random.Random(SEED).shuffle(unsafe_ids)

models, condition_rows, assignments = {}, [], []

for ratio in RATIOS:
    count = int(ratio * len(unsafe_ids))
    rewritten_ids = set(unsafe_ids[:count])
    use_rewrite = train.case_id.isin(rewritten_ids)

    texts = train["original_text"].copy()
    texts.loc[use_rewrite] = train.loc[
        use_rewrite, "rewritten_text"
    ]

    model = make_pipeline(
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=15000,
            sublinear_tf=True
        ),
        LogisticRegression(
            solver="liblinear",
            max_iter=1000,
            random_state=SEED
        )
    )

    model.fit(texts, train["label"])
    models[ratio] = model

    condition_rows.append({
        "rewrite_ratio": ratio,
        "training_cases": len(train),
        "unsafe_cases": len(unsafe_ids),
        "rewritten_unsafe": count
    })

    assignments.extend({
        "ratio": ratio,
        "case_id": case,
        "version": (
            "rewritten" if case in rewritten_ids else "original"
        )
    } for case in train.case_id)

conditions = pd.DataFrame(condition_rows)
display(conditions)

## 6. Metrics and validation evaluation

Unsafe recall = TP / (TP + FN).
Safe false-positive rate = FP / (FP + TN).

The rewritten test view replaces unsafe texts only. Safe texts remain identical.

The always-safe baseline has recall 0 and false-positive rate 0. This illustrates
why a low false-positive rate alone is not sufficient.

No tuning is performed. The threshold remains 0.5.

In [ ]:
def evaluate(frame, split_name):
    summaries, predictions = [], []

    for view in ["original", "rewritten"]:
        texts = frame["original_text"].copy()
        unsafe = frame.label == 1

        if view == "rewritten":
            texts.loc[unsafe] = frame.loc[
                unsafe, "rewritten_text"
            ]

        for ratio in [None] + RATIOS:
            if ratio is None:
                scores = np.zeros(len(frame))
            else:
                scores = models[ratio].predict_proba(texts)[:, 1]

            predicted = (scores >= THRESHOLD).astype(int)

            tn, fp, fn, tp = confusion_matrix(
                frame.label, predicted, labels=[0, 1]
            ).ravel()

            model_name = (
                "always_safe" if ratio is None else "tfidf_lr"
            )

            summaries.append({
                "split": split_name,
                "model": model_name,
                "rewrite_ratio": ratio,
                "view": view,
                "unsafe_recall": tp / (tp + fn),
                "safe_fpr": fp / (fp + tn),
                "auc": roc_auc_score(frame.label, scores),
                "tp": int(tp),
                "fn": int(fn),
                "fp": int(fp),
                "tn": int(tn)
            })

            predictions.extend({
                "case_id": case,
                "model": model_name,
                "rewrite_ratio": ratio,
                "view": view,
                "label": int(label),
                "unsafe_score": float(score),
                "prediction": int(prediction)
            } for case, label, score, prediction in zip(
                frame.case_id, frame.label, scores, predicted
            ))

    return pd.DataFrame(summaries), pd.DataFrame(predictions)

validation_results, validation_predictions = evaluate(
    splits["validation"], "validation"
)

display(validation_results.round(4))
print("The threshold remains fixed at 0.5 for every condition.")

## 7. Evaluate the fixed test split

Evaluate all predeclared conditions without selecting a winner or changing
the threshold.

The main outcomes are unsafe recall and safe false-positive rate. AUC is
secondary.

Safe FPR must match between the two views because the safe texts are identical.

In [ ]:
test_results, test_predictions = evaluate(
    splits["test"], "test"
)

lr_results = test_results[
    test_results.model == "tfidf_lr"
]

assert (
    lr_results.groupby("rewrite_ratio")["safe_fpr"]
    .nunique().eq(1).all()
)

assert (test_results.tp + test_results.fn).eq(535).all()
assert (test_results.fp + test_results.tn).eq(535).all()

display(test_results.round(4))

## 8. Plot the pilot

All rates are percentages.

The left panel shows recall for each unsafe text version.
The right panel shows one FPR curve because safe texts do not change.

Only three ratios are tested. Connecting lines are visual guides.

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 4),
    constrained_layout=True
)

for view, color, marker in [
    ("original", "#0072B2", "o"),
    ("rewritten", "#D55E00", "s")
]:
    part = (
        lr_results[lr_results.view == view]
        .sort_values("rewrite_ratio")
    )

    axes[0].plot(
        part.rewrite_ratio * 100,
        part.unsafe_recall * 100,
        color=color,
        marker=marker,
        label=f"{view.title()} unsafe text"
    )

part = (
    lr_results[lr_results.view == "original"]
    .sort_values("rewrite_ratio")
)

axes[1].plot(
    part.rewrite_ratio * 100,
    part.safe_fpr * 100,
    color="#009E73",
    marker="D"
)

axes[0].set(
    title="Unsafe recall: higher is better",
    ylabel="Unsafe recall (%)"
)

axes[1].set(
    title="Safe false-positive rate: lower is better",
    ylabel="Safe false-positive rate (%)"
)

for ax in axes:
    ax.set(
        xlabel="Unsafe training cases replaced (%)",
        xticks=[0, 50, 100],
        ylim=(-2, 102)
    )
    ax.grid(alpha=0.25)

axes[0].legend(fontsize=9)
fig.suptitle("PolyGuard finance pilot | Fixed threshold = 0.5")

fig.savefig(
    RESULTS / "pilot_results.png",
    dpi=200
)

plt.show()

## 9. Save Reproducible Outputs

The ZIP contains:

- Data files for manual Hugging Face upload.
- Pilot results and configuration files for manual GitHub upload.

No remote repository is changed.

Write the dataset card and your interpretation yourself, following the course Human-Only requirement.

Source: [PolyGuard dataset](https://huggingface.co/datasets/Virtue-AI-HUB/PolyGuard), CC BY 4.0.

Cite the [associated NeurIPS 2025 paper](https://proceedings.nips.cc/paper_files/paper/2025/hash/11ed9cdc955e23684a1beae9cb8da059-Abstract-Datasets_and_Benchmarks_Track.html).

Before interpreting results, consider:

- Which condition misses fewer unsafe cases?
- Does it increase false alarms?
- Are differences stable across repeated splits?

Do not infer real-world reliability from this single synthetic-data pilot.

In [ ]:
for name, table in [
    ("validation_results", validation_results),
    ("test_results", test_results),
    ("validation_predictions", validation_predictions),
    ("test_predictions", test_predictions),
    ("training_conditions", conditions),
    ("training_assignments", pd.DataFrame(assignments))
]:
    table.to_csv(
        RESULTS / f"{name}.csv",
        index=False
    )

provenance = {
    "source_repo": REPO,
    "source_revision": REVISION,
    "source_license": "cc-by-4.0",
    "seed": SEED,
    "held_out_unit": "normalized rule",
    "input_files": raw_manifest,
    "split_counts": {
        name: {
            "cases": len(frame),
            "rules": int(frame.rule_id.nunique())
        }
        for name, frame in splits.items()
    },
    "limitations": [
        "Synthetic examples",
        "No expert label audit",
        "Exploratory split"
    ]
}

(DATA / "provenance.json").write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8"
)

config = {
    "ratios": RATIOS,
    "seed": SEED,
    "threshold": THRESHOLD,
    "model": models[0.0].get_params()[
        "logisticregression"
    ].get_params(),
    "tfidf": {
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_features": 15000,
        "sublinear_tf": True
    },
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "requests": requests.__version__
}

(RESULTS / "experiment_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

archive = ROOT.parent / "polyguard_colab_outputs"
staging = ROOT / "export"

for name, source in [
    ("data", DATA),
    ("results", RESULTS)
]:
    shutil.copytree(
        source,
        staging / name,
        dirs_exist_ok=True
    )

archive_path = shutil.make_archive(
    str(archive),
    "zip",
    root_dir=staging
)

print("Saved:", archive_path)
print(
    "Hugging Face: manually upload data/*.jsonl "
    "and data/provenance.json."
)
print(
    "GitHub: manually upload this notebook "
    "and the pilot results."
)

## 10. Download the Output Archive

In Colab, this cell downloads the ZIP to your computer.

Also use File > Download > Download .ipynb to save the notebook with your own executed outputs.

Uploading files to the course repositories remains a manual step.

In [ ]:
try:
    from google.colab import files
except ImportError:
    print(
        "Not running in Colab. "
        "The archive is available at:",
        archive_path
    )
else:
    files.download(archive_path)